
# SCE-Net: переход от triplet-learning к pair-classification

Этот ноутбук **переиспользует максимум кода** из `sce_net_fashion_compatibility.ipynb`,
но меняет постановку задачи:

- раньше: `triplet loss` на `(anchor, positive, negative)`;
- теперь: **бинарная классификация пар**.

Цели:
1. Из уже готовых `train_triplets` и `val_triplets` собрать пары с таргетами:
   - `(anchor, positive) -> y=1`
   - `(anchor, negative) -> y=0`
2. Обучить модель-классификатор на основе SCE-Net представлений.
3. Сравнить метрики (ROC-AUC/PR-AUC/F1/LogLoss) с триплетным вариантом.

---

## Важное условие
Код начинается с точки, где переменные **`train_triplets`** и **`val_triplets`** уже существуют в памяти.


In [ ]:

# Если вы запускаете этот ноутбук отдельно, убедитесь, что train_triplets и val_triplets уже определены.
required_vars = ['train_triplets', 'val_triplets']
for v in required_vars:
    if v not in globals():
        raise RuntimeError(f'Variable `{v}` is not defined. Запустите шаг построения триплетов заранее.')

print('train_triplets:', train_triplets.shape)
print('val_triplets:', val_triplets.shape)


## 1) Импорты и базовые настройки


In [ ]:

import random
import time
from dataclasses import dataclass
from typing import Dict

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, log_loss, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 2) Конфиг для pair-classification


In [ ]:

@dataclass
class PairClsConfig:
    hf_model_name: str = 'openai/clip-vit-base-patch32'

    # Архитектура SCE-Net части
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1

    # Классификатор пары (голова поверх эмбеддингов)
    pair_hidden: int = 1024

    # Оптимизация
    batch_size: int = 64
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 6
    use_amp: bool = True
    freeze_backbone: bool = False

    # Регуляризация из SCE-Net
    lambda_l1: float = 1e-4
    lambda_l2: float = 1e-4

    # DataLoader
    num_workers: int = 4
    prefetch_factor: int = 2
    persistent_workers: bool = True

cfg = PairClsConfig()
cfg


## 3) Из триплетов в пары для классификации (y=1/0)


In [ ]:

def triplets_to_pair_df(triplets_df: pd.DataFrame) -> pd.DataFrame:
    """
    Превращаем каждый triplet в 2 пары:
      (anchor, positive) -> y=1
      (anchor, negative) -> y=0

    Важно: в исходных triplets одна и та же positive-пара может повторяться
    много раз (из-за нескольких sampled negative), поэтому здесь удаляем
    дубликаты пар по (sku1, sku2, y, sku1_path, sku2_path).
    """
    pos_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['pos_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['pos_path'].astype(str),
        'y': 1,
    })
    neg_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['neg_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['neg_path'].astype(str),
        'y': 0,
    })

    pair_df = pd.concat([pos_df, neg_df], axis=0, ignore_index=True)

    # Удаляем дубликаты и только затем перемешиваем.
    pair_df = pair_df.drop_duplicates(
        subset=['sku1', 'sku2', 'sku1_path', 'sku2_path', 'y'],
        keep='first'
    ).reset_index(drop=True)

    pair_df = pair_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return pair_df

train_pairs = triplets_to_pair_df(train_triplets)
val_pairs = triplets_to_pair_df(val_triplets)

print('train_pairs:', train_pairs.shape, '| pos_rate=', train_pairs['y'].mean())
print('val_pairs  :', val_pairs.shape, '| pos_rate=', val_pairs['y'].mean())
train_pairs.head()


## 4) RAM-cache изображений (переиспользуем подход из исходного ноутбука)


In [ ]:

def build_image_cache(paths, verbose: bool = True):
    unique_paths = sorted(set(map(str, paths)))
    cache = {}

    iterator = tqdm(unique_paths, desc='Caching images to RAM') if verbose else unique_paths
    for p in iterator:
        img_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'Cannot read image while caching: {p}')
        cache[p] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if verbose:
        total_mb = sum(v.nbytes for v in cache.values()) / (1024 ** 2)
        print(f'Cached images: {len(cache)} | approx RAM: {total_mb:.1f} MB')
    return cache

all_paths = pd.concat([
    train_pairs['sku1_path'], train_pairs['sku2_path'],
    val_pairs['sku1_path'], val_pairs['sku2_path'],
], axis=0).astype(str).tolist()

IMAGE_CACHE = build_image_cache(all_paths, verbose=True)


## 5) Dataset / Collate для пары + бинарный таргет


In [ ]:

class PairImageDataset(Dataset):
    def __init__(self, pair_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = pair_df.reset_index(drop=True)
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def _load_img(self, p: str):
        p = str(p)
        if p not in self.image_cache:
            raise KeyError(f'Image path not found in cache: {p}')
        return self.image_cache[p]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img1 = self._load_img(row['sku1_path'])
        img2 = self._load_img(row['sku2_path'])
        y = np.float32(row['y'])
        return img1, img2, y


def make_pair_collate(processor: CLIPProcessor):
    def collate_fn(batch):
        imgs1, imgs2, ys = zip(*batch)
        px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']
        px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']
        ys = torch.tensor(ys, dtype=torch.float32)
        return px1, px2, ys
    return collate_fn

processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)

train_ds = PairImageDataset(train_pairs, image_cache=IMAGE_CACHE)
val_ds = PairImageDataset(val_pairs, image_cache=IMAGE_CACHE)

collate_fn = make_pair_collate(processor)

loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if cfg.num_workers > 0:
    loader_kwargs['prefetch_factor'] = cfg.prefetch_factor
    loader_kwargs['persistent_workers'] = cfg.persistent_workers

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

print('train batches:', len(train_loader), 'val batches:', len(val_loader))


## 6) SCE-Net encoder + классификатор пары


In [ ]:

class SCENetEncoder(nn.Module):
    """
    Переиспользуем SCE-Net механику (condition masks + weight branch),
    но делаем универсальный encoder для одной картинки и для пары.
    """
    def __init__(self, clip_model_name: str, num_conditions: int = 5, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            emb_dim = proj.out_features if hasattr(proj, 'out_features') else self.clip.config.projection_dim
        else:
            emb_dim = getattr(self.clip.config, 'projection_dim', None)
            if emb_dim is None:
                raise ValueError('Cannot infer embedding dimension D from model config/projection.')

        self.emb_dim = emb_dim
        self.num_conditions = num_conditions

        self.condition_masks = nn.Parameter(torch.empty(num_conditions, emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)

        self.weight_branch = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions)
        )

    def _extract_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        if hasattr(self.clip, 'get_image_features'):
            out = self.clip.get_image_features(pixel_values=pixel_values)
            if torch.is_tensor(out):
                return out
            if hasattr(out, 'image_embeds') and out.image_embeds is not None:
                return out.image_embeds
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                return out.pooler_output
            if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
                return out.last_hidden_state[:, 0, :]

        out = self.clip(pixel_values=pixel_values, return_dict=True)
        if hasattr(out, 'image_embeds') and out.image_embeds is not None:
            return out.image_embeds
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            return out.last_hidden_state[:, 0, :]
        raise RuntimeError('CLIP output does not contain image features in expected fields.')

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        feats = self._extract_image_features(pixel_values)
        return F.normalize(feats, p=2, dim=-1)

    def forward_pair_embeddings(self, px1: torch.Tensor, px2: torch.Tensor):
        v1 = self.encode_image(px1)
        v2 = self.encode_image(px2)

        c = torch.sigmoid(self.condition_masks)  # [M, D]
        o1 = c.unsqueeze(0) * v1.unsqueeze(1)    # [B, M, D]
        o2 = c.unsqueeze(0) * v2.unsqueeze(1)    # [B, M, D]

        pair_cat = torch.cat([v1, v2], dim=-1)   # [B, 2D]
        w_logits = self.weight_branch(pair_cat)   # [B, M]
        w = F.softmax(w_logits, dim=-1)           # [B, M]

        e1 = torch.einsum('bm,bmd->bd', w, o1)
        e2 = torch.einsum('bm,bmd->bd', w, o2)
        return e1, e2


class SCEPairClassifier(nn.Module):
    """
    Классификация пары: logits -> BCEWithLogitsLoss.
    Фичи пары: [e1, e2, |e1-e2|, e1*e2].
    """
    def __init__(self, cfg: PairClsConfig):
        super().__init__()
        self.encoder = SCENetEncoder(
            clip_model_name=cfg.hf_model_name,
            num_conditions=cfg.num_conditions,
            hidden_dim=cfg.condition_hidden,
            dropout=cfg.dropout,
        )
        d = self.encoder.emb_dim
        self.classifier = nn.Sequential(
            nn.Linear(4 * d, cfg.pair_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.pair_hidden, 1),
        )

    def forward(self, px1: torch.Tensor, px2: torch.Tensor):
        e1, e2 = self.encoder.forward_pair_embeddings(px1, px2)
        feat = torch.cat([e1, e2, torch.abs(e1 - e2), e1 * e2], dim=-1)
        logit = self.classifier(feat).squeeze(-1)
        return logit, e1, e2

model = SCEPairClassifier(cfg).to(device)

if cfg.freeze_backbone:
    for p in model.encoder.clip.parameters():
        p.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable params:', trainable_params)
print('embedding dim D:', model.encoder.emb_dim)


## 7) Loss и метрики


In [ ]:

def pair_classification_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    e1: torch.Tensor,
    e2: torch.Tensor,
    condition_masks: torch.Tensor,
    lambda_l1: float,
    lambda_l2: float,
):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    l1 = condition_masks.abs().mean()
    l2 = (e1.pow(2).mean() + e2.pow(2).mean()) / 2.0
    total = bce + lambda_l1 * l1 + lambda_l2 * l2

    return total, {
        'loss': total.item(),
        'bce': bce.item(),
        'l1': l1.item(),
        'l2': l2.item(),
    }


def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, thr: float = 0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'f1@0.5': f1_score(y_true, y_pred),
        'acc@0.5': accuracy_score(y_true, y_pred),
        'logloss': log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
    }


## 8) Train/Val loop для pair-classification


In [ ]:

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))


def run_epoch_pair_cls(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    losses = []
    probs_all, y_all = [], []

    data_time = 0.0
    compute_time = 0.0
    t_prev = time.perf_counter()

    for px1, px2, y in tqdm(loader, leave=False):
        t_after_load = time.perf_counter()
        data_time += (t_after_load - t_prev)

        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        t0 = time.perf_counter()
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                logits, e1, e2 = model(px1, px2)
                loss, stats = pair_classification_loss(
                    logits=logits,
                    targets=y,
                    e1=e1,
                    e2=e2,
                    condition_masks=model.encoder.condition_masks,
                    lambda_l1=cfg.lambda_l1,
                    lambda_l2=cfg.lambda_l2,
                )

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_time += (time.perf_counter() - t0)

        losses.append(stats['loss'])

        prob = torch.sigmoid(logits).detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        probs_all.append(prob)
        y_all.append(y_np)

        t_prev = time.perf_counter()

    y_prob = np.concatenate(probs_all)
    y_true = np.concatenate(y_all).astype(int)
    metrics = compute_binary_metrics(y_true, y_prob, thr=0.5)
    metrics['loss'] = float(np.mean(losses))
    metrics['data_time_sec'] = data_time
    metrics['compute_time_sec'] = compute_time
    return metrics


## 9) Обучение


In [ ]:

history = []
best_state = None
best_score = -np.inf

for epoch in range(1, cfg.epochs + 1):
    tr = run_epoch_pair_cls(model, train_loader, optimizer=optimizer)
    va = run_epoch_pair_cls(model, val_loader, optimizer=None)

    row = {
        'epoch': epoch,
        'train_loss': tr['loss'],
        'val_loss': va['loss'],
        'train_roc_auc': tr['roc_auc'],
        'val_roc_auc': va['roc_auc'],
        'train_pr_auc': tr['pr_auc'],
        'val_pr_auc': va['pr_auc'],
        'val_f1@0.5': va['f1@0.5'],
        'val_acc@0.5': va['acc@0.5'],
        'val_logloss': va['logloss'],
    }
    history.append(row)

    score = va['pr_auc']
    if score > best_score:
        best_score = score
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"[Epoch {epoch:02d}] "
        f"train loss={tr['loss']:.4f} auc={tr['roc_auc']:.4f} pr={tr['pr_auc']:.4f} | "
        f"val loss={va['loss']:.4f} auc={va['roc_auc']:.4f} pr={va['pr_auc']:.4f} "
        f"f1={va['f1@0.5']:.4f}"
    )

hist_df = pd.DataFrame(history)
hist_df


## 10) Загрузка лучшей эпохи + подбор порога


In [ ]:

if best_state is not None:
    model.load_state_dict(best_state)

@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs, ys = [], []
    for px1, px2, y in tqdm(loader, leave=False):
        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        logits, _, _ = model(px1, px2)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys).astype(int)

val_prob, val_true = predict_proba(model, val_loader)

thresholds = np.linspace(0.1, 0.9, 81)
rows = []
for thr in thresholds:
    pred = (val_prob >= thr).astype(int)
    rows.append({'thr': thr, 'f1': f1_score(val_true, pred), 'acc': accuracy_score(val_true, pred)})

thr_df = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
best_thr = float(thr_df.loc[0, 'thr'])

print('Best threshold by F1 on val:', best_thr)
print(thr_df.head(10))



## 11) Как корректно сравнить с triplet-обучением

Чтобы честно ответить на вопрос **«может ли это улучшить метрику?»**, нужно сделать A/B:

1. Зафиксировать одинаковые split'ы (`train_df`, `val_df`, `test_df`) и одинаковый CLIP backbone.
2. Обучить:
   - baseline: triplet objective (как в старом ноутбуке),
   - candidate: pair classification (этот ноутбук).
3. Считать метрики на одном и том же `test_df`:
   - ROC-AUC,
   - PR-AUC,
   - Recall@K / Precision@K (если задача retrieval),
   - calibration (logloss/Brier), если важны вероятности.

### Когда классификация может выиграть
- Если нужна именно **вероятность совместимости** (а не только ranking-distance).
- Если данные шумные: BCE часто даёт более стабильный градиент, чем triplet hinge.
- Если есть class imbalance и вы добавите `pos_weight` в BCE.

### Когда triplet может быть лучше
- Если главный KPI — ranking/retrieval, где margin-based обучение часто сильнее.
- Если негативы сложные (hard negatives) и хорошо подобраны.

### Практический вывод
Да, **метрика может улучшиться**, но это эмпирический вопрос.
Этот ноутбук даёт готовый пайплайн для эксперимента с минимальными изменениями относительно исходного кода.
